# 06 – Geography of entire homes accepting short stays
## Question
Which neighbourhoods have the highest share of entire-home listings with a minimum stay of 1–29 nights?
Variables: `room_type`, `minimum_nights`, `neighbourhood_cleansed`, and the repository GeoJSON polygons.
The denominator is **all Airbnb listings in each neighbourhood**, including missing prices. Neighbourhoods with fewer than 20 listings are flagged and not coloured. These are neighbourhoods (barris/barrios), not the larger administrative districts. No external housing-stock denominator is used.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src.study_utils import load_spain, export_table, save_figure, SPAIN
from src.cities import CITIES

data = load_spain()
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from src.study_utils import neighbourhood_map
from src.load import load_neighbourhoods


In [2]:
summary = data.groupby(['city', 'neighbourhood_cleansed']).agg(
    listings=('id', 'size'), short_entire=('short_entire', 'sum')).reset_index()
summary['share_pct'] = 100 * summary.short_entire / summary.listings
summary['small_sample'] = summary.listings < 20
assert (summary.short_entire <= summary.listings).all()
assert summary.listings.sum() == len(data)
export_table(summary, 'study06_neighbourhoods')
top = (summary[~summary.small_sample].sort_values(['city', 'share_pct', 'listings'], ascending=[True, False, False])
       .groupby('city').head(5))
display(export_table(top, 'study06_top_neighbourhoods'))
city_summary = data.groupby('city').agg(listings=('id', 'size'), short_entire=('short_entire', 'sum'))
city_summary['share_pct'] = 100 * city_summary.short_entire / city_summary.listings
display(export_table(city_summary, 'study06_city_summary', index=True))


,city,neighbourhood_cleansed,listings,short_entire,share_pct,small_sample
24,barcelona,Vilapicina i la Torre Llobeta,79,57,72.151899,False
38,barcelona,el Poble Sec,650,425,65.384615,False
2,barcelona,Diagonal Mar i el Front Marítim del Poblenou,119,77,64.705882,False
56,barcelona,la Sagrada Família,910,581,63.846154,False
11,barcelona,Sant Antoni,846,494,58.392435,False
110,madrid,Costillares,30,27,90.000000,False
170,madrid,Rejas,144,115,79.861111,False
136,madrid,Legazpi,44,35,79.545455,False
150,madrid,Orcasur,66,51,77.272727,False
179,madrid,San Juan Bautista,29,22,75.862069,False


,listings,short_entire,share_pct
city,,,
barcelona,15293,7069,46.223763
madrid,22708,13903,61.225119


In [3]:
joins = []
for city in SPAIN:
    table = summary[summary.city.eq(city)].set_index('neighbourhood_cleansed')
    names = {f['properties']['neighbourhood'] for f in load_neighbourhoods(city)['features']}
    unmatched = set(table.index) - names
    assert not unmatched, unmatched
    joins.append({'city': city, 'listing_neighbourhoods': len(table), 'polygon_names': len(names),
                  'unmatched': len(unmatched), 'small_sample': int(table.small_sample.sum()),
                  'polygons_without_listings': len(names - set(table.index))})
    neighbourhood_map(city, table, 'share_pct', 'entire homes, minimum 1–29 nights',
                      f'study06_short_entire_{city}')
    plt.show()
display(export_table(pd.DataFrame(joins), 'study06_join_audit'))


/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11852/3357864750.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11852/3357864750.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,city,listing_neighbourhoods,polygon_names,unmatched,small_sample,polygons_without_listings
0,barcelona,69,73,0,18,4
1,madrid,128,128,0,17,0


## Findings and limitations
The tables above rank the neighbourhoods using the same definition and the two maps share a fixed 0–100% scale. A high share describes the composition of Airbnb supply, not the share of the local housing stock. A minimum of 1–29 nights permits short stays but does not show the actual length of bookings. Listings are not necessarily unique dwellings, available units, or homes removed from residential use. Smaller samples remain in the exported table with a flag; their values are not coloured or included in the ranking.


In this snapshot the segment contains 7,069 listings in Barcelona (46.2% of all listings) and 13,903 in Madrid (61.2%). The leading eligible neighbourhoods are Vilapicina i la Torre Llobeta (72.2%, 57/79) and Costillares (90.0%, 27/30). The latter has a modest denominator even after the threshold. Barcelona's 75 GeoJSON features represent 73 distinct names because one neighbourhood has three polygon features; the join uses distinct names, not feature counts.